In [ ]:
!fusermount -u /content/drive  # unmount if mounted
!rm -rf /content/drive         # remove any leftover files

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, json
from pathlib import Path
from PIL import Image

ROOT = Path("/content/drive/MyDrive/Store/MOT20/MOT20-clean")
OUT_JSON = ROOT / "coco_format"
OUT_JSON.mkdir(parents=True, exist_ok=True)

splits = ["train", "val"]
categories = [{"id": 1, "name": "person"}]

for split in splits:
    im_root = ROOT / "images" / split
    lb_root = ROOT / "labels" / split

    coco_dict = {"images": [], "annotations": [], "categories": categories}
    ann_id = 1
    img_id = 1

    print(f"\nProcessing split: {split}")

    for seq in sorted(im_root.glob("*")):
        print(f"  Sequence: {seq.name}")
        for img_path in sorted(seq.glob("*")):
            if not img_path.is_file():
                continue
            try:
                with Image.open(img_path) as im:
                    w, h = im.size
            except Exception as e:
                print("  Skipping corrupt image:", img_path, e)
                continue

            # Add image entry
            coco_dict["images"].append({
                "id": img_id,
                "file_name": f"{seq.name}/{img_path.name}",
                "height": h,
                "width": w
            })

            # Matching YOLO label
            lb_file = lb_root / seq.name / f"{img_path.stem}.txt"
            if lb_file.exists():
                with open(lb_file) as f:
                    for line in f:
                        cls, x, y, bw, bh = map(float, line.strip().split())
                        x1 = (x - bw/2) * w
                        y1 = (y - bh/2) * h
                        bw_px = bw * w
                        bh_px = bh * h
                        coco_dict["annotations"].append({
                            "id": ann_id,
                            "image_id": img_id,
                            "category_id": 1,
                            "bbox": [x1, y1, bw_px, bh_px],
                            "area": bw_px * bh_px,
                            "iscrowd": 0
                        })
                        ann_id += 1
            else:
                print("  No label for:", img_path.name)

            img_id += 1

    out_file = OUT_JSON / f"{split}.json"
    with open(out_file, "w") as f:
        json.dump(coco_dict, f, indent=2)

    print(f"Saved {split}.json → {out_file}")
    print(f"   Images: {len(coco_dict['images'])}, Annotations: {len(coco_dict['annotations'])}")

print("\nConversion finished. COCO JSONs are in:", OUT_JSON)



Processing split: train
  Sequence: MOT20-01
  Skipping corrupt image: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000002.jpg cannot identify image file '/content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000002.jpg'
  Skipping corrupt image: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000004.jpg cannot identify image file '/content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000004.jpg'
  Skipping corrupt image: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000007.jpg cannot identify image file '/content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000007.jpg'
  Skipping corrupt image: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000009.jpg cannot identify image file '/content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000009.jpg'
  Skipping corrupt image: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01

In [ ]:
from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/Store/MOT20/MOT20-clean")
OUT_JSON = ROOT / "coco_format"

with open(OUT_JSON / "train.json") as f:
    train_json = json.load(f)
with open(OUT_JSON / "val.json") as f:
    val_json = json.load(f)

print("Loaded COCO JSONs:", list(OUT_JSON.glob("*.json")))


Loaded COCO JSONs: [PosixPath('/content/drive/MyDrive/Store/MOT20/MOT20-clean/coco_format/train.json'), PosixPath('/content/drive/MyDrive/Store/MOT20/MOT20-clean/coco_format/val.json')]


In [ ]:
# INSTALLS
!pip -q install --upgrade transformers timm

# LOAD MODEL & PROCESSOR
import torch, os
from transformers import AutoImageProcessor, DeformableDetrForObjectDetection

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_ID = "SenseTime/deformable-detr"  # COCO-pretrained R50
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = DeformableDetrForObjectDetection.from_pretrained(MODEL_ID).to(DEVICE).eval()

# Find the 'person' class id from the model config
id2label = model.config.id2label
label2id = {v: k for k, v in id2label.items()}
PERSON_ID = label2id.get("person", 1)  # COCO person is usually 1
print("Loaded", MODEL_ID, "on", DEVICE, "| person_id =", PERSON_ID)



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 643.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 44.6 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/305 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/161M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pas

Loaded SenseTime/deformable-detr on cpu | person_id = 1


In [ ]:
import os, json
from pathlib import Path
from PIL import Image
import torch
from transformers import AutoImageProcessor, DeformableDetrForObjectDetection
from collections import Counter

# --- Load pretrained Deformable DETR ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "SenseTime/deformable-detr"
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = DeformableDetrForObjectDetection.from_pretrained(MODEL_ID).to(DEVICE).eval()

id2label = model.config.id2label
label2id = {v: k for k, v in id2label.items()}
PERSON_ID = label2id.get("person", 1)

# --- Paths ---
DATA_ROOT = Path("/content/drive/MyDrive/Store/MOT20/MOT20-clean")
SEQS = {
    "train": ["MOT20-01", "MOT20-02", "MOT20-03"],
    "val": ["MOT20-05"],
}
OUT_DIR = Path("/content/deform_detr_dets2")
OUT_DIR.mkdir(parents=True, exist_ok=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary t

In [ ]:
# --- Collect images ---
all_imgs = []
for split, seqs in SEQS.items():
    for seq in seqs:
        seq_dir = DATA_ROOT / "images" / split / seq   # ✅ no img1
        if not seq_dir.exists():
            print(f"⚠️ Skipping {seq}, folder not found: {seq_dir}")
            continue
        for p in sorted(seq_dir.rglob("*.jpg")):
            try:
                with Image.open(p) as im:
                    im.verify()  # check if image is valid
                all_imgs.append(p)
            except Exception:
                print(f"⚠️ Bad image skipped: {p}")

print(f"✅ Found {len(all_imgs)} images across splits {list(SEQS.keys())}")


⚠️ Bad image skipped: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000002.jpg
⚠️ Bad image skipped: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000004.jpg
⚠️ Bad image skipped: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000007.jpg
⚠️ Bad image skipped: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000009.jpg
⚠️ Bad image skipped: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000011.jpg
⚠️ Bad image skipped: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000015.jpg
✅ Found 8925 images across splits ['train', 'val']


In [ ]:
from collections import Counter

# --- Inference ---
BATCH_SIZE = 8

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

processed = 0
all_pred_labels = []

for batch_paths in chunks(all_imgs, BATCH_SIZE):
    images = [Image.open(p).convert("RGB") for p in batch_paths]
    sizes_hw = torch.tensor([[im.height, im.width] for im in images], device=DEVICE)
    inputs = processor(images=images, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_object_detection(outputs, target_sizes=sizes_hw, threshold=0.05)

    for img_path, im, res in zip(batch_paths, images, results):
        # Collect all predicted labels for debugging
        all_pred_labels.extend(res["labels"].tolist())

        dets = []
        for score, label, box in zip(res["scores"], res["labels"], res["boxes"]):
            if int(label) != PERSON_ID:
                continue
            x1, y1, x2, y2 = box.tolist()
            dets.append([float(x1), float(y1), float(x2), float(y2), float(score), 0])  # cls=0 for person

        # save YOLO-style per-image JSON
        seq_name = img_path.parent.name
        frame_name = img_path.stem
        out_name = f"{seq_name}_{frame_name}.json"
        with open(OUT_DIR / out_name, "w") as f:
            json.dump(dets, f)

        im.close()

    processed += len(batch_paths)
    if processed % (BATCH_SIZE * 10) == 0 or processed == len(all_imgs):
        print(f"Processed {processed}/{len(all_imgs)} images")

# --- Debug output ---
print("Predicted label counts:", Counter(all_pred_labels))

# --- Copy results to Drive ---
drive_out = Path("/content/drive/MyDrive/Store/deform_detr_dets")
os.system(f"rm -rf {drive_out} && mkdir -p {drive_out} && cp -r {OUT_DIR}/* {drive_out}/")
print("Copied results to:", drive_out)


In [ ]:
# --- Inference ---
BATCH_SIZE = 2   # keep small for CPU

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

processed = 0
all_pred_labels = []

for batch_paths in chunks(all_imgs, BATCH_SIZE):
    # Load images safely, resize for CPU efficiency
    images = []
    for p in batch_paths:
        im = Image.open(p).convert("RGB")
        im.thumbnail((800, 800))   #  downscale for memory safety
        images.append(im)

    sizes_hw = torch.tensor([[im.height, im.width] for im in images], device=DEVICE)
    inputs = processor(images=images, return_tensors="pt").to(DEVICE)

    with torch.inference_mode():
        outputs = model(**inputs)

    results = processor.post_process_object_detection(outputs, target_sizes=sizes_hw, threshold=0.05)

    for img_path, im, res in zip(batch_paths, images, results):
        all_pred_labels.extend(res["labels"].tolist())

        dets = []
        for score, label, box in zip(res["scores"], res["labels"], res["boxes"]):
            if int(label) != PERSON_ID:
                continue
            x1, y1, x2, y2 = box.tolist()
            dets.append([float(x1), float(y1), float(x2), float(y2), float(score), 0])  # cls=0 for person

        # --- save per-image JSON inside seq folder ---
        seq_name = img_path.parent.parent.name  # MOT20-01, MOT20-02, etc.
        frame_name = img_path.stem
        seq_out_dir = OUT_DIR / seq_name
        seq_out_dir.mkdir(parents=True, exist_ok=True)

        out_path = seq_out_dir / f"{frame_name}.json"
        with open(out_path, "w") as f:
            json.dump(dets, f)

        im.close()

    # cleanup after batch
    del inputs, outputs, results, images
    torch.cuda.empty_cache()  # harmless on CPU

    processed += len(batch_paths)
    if processed % (BATCH_SIZE * 10) == 0 or processed == len(all_imgs):
        print(f"Processed {processed}/{len(all_imgs)} images")

# --- Debug output ---
print("Predicted label counts:", Counter(all_pred_labels))

# --- Copy results to Drive ---
drive_out = Path("/content/drive/MyDrive/Store/deform_detr_dets2")   #  new folder in Drive
os.system(f"rm -rf {drive_out} && mkdir -p {drive_out}")
os.system(f"cp -r {OUT_DIR}/* {drive_out}/")  #  this will copy subfolders too
print("Copied results to:", drive_out)

Processed 20/8925 images
Processed 40/8925 images
Processed 60/8925 images
Processed 80/8925 images
Processed 100/8925 images
Processed 120/8925 images
Processed 140/8925 images
Processed 160/8925 images
Processed 180/8925 images
Processed 200/8925 images
Processed 220/8925 images
Processed 240/8925 images
Processed 260/8925 images
Processed 280/8925 images
Processed 300/8925 images
Processed 320/8925 images
Processed 340/8925 images
Processed 360/8925 images
Processed 380/8925 images
Processed 400/8925 images
Processed 420/8925 images
Processed 440/8925 images
Processed 460/8925 images
Processed 480/8925 images
Processed 500/8925 images
Processed 520/8925 images
Processed 540/8925 images
Processed 560/8925 images
Processed 580/8925 images
Processed 600/8925 images
Processed 620/8925 images
Processed 640/8925 images
Processed 660/8925 images
Processed 680/8925 images
Processed 700/8925 images
Processed 720/8925 images
Processed 740/8925 images
Processed 760/8925 images
Processed 780/89

In [ ]:
import json
from pathlib import Path
from PIL import Image
import numpy as np

# IoU helper
def box_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area1 = (box1[2]-box1[0])*(box1[3]-box1[1])
    area2 = (box2[2]-box2[0])*(box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter/union if union>0 else 0

def evaluate_detr(pred_dir, im_root, lb_root, iou_thr=0.5, conf_thr=0.25):
    tp, fp, fn = 0, 0, 0
    all_scores, all_matches = [], []

    for pred_file in sorted(pred_dir.glob("*.json")):
        preds = json.load(open(pred_file))
        preds = [p for p in preds if p[4] > conf_thr]   # keep high conf
        pred_boxes = [[p[0], p[1], p[2], p[3], p[4]] for p in preds]

        seq, frame = pred_file.stem.split("_", 1)
        img_path = im_root / seq / f"{frame}.jpg"
        w, h = Image.open(img_path).size

        # Load GT (YOLO format)
        gt_file = lb_root / seq / f"{frame}.txt"
        gt_boxes = []
        if gt_file.exists():
            for line in open(gt_file):
                cls, x, y, bw, bh = map(float, line.strip().split())
                x1 = (x - bw/2) * w
                y1 = (y - bh/2) * h
                x2 = (x + bw/2) * w
                y2 = (y + bh/2) * h
                gt_boxes.append([x1,y1,x2,y2])

        matched = set()
        for pb in pred_boxes:
            found = False
            for gi, gb in enumerate(gt_boxes):
                if gi in matched:
                    continue
                if box_iou(pb[:4], gb) >= iou_thr:
                    tp += 1
                    matched.add(gi)
                    found = True
                    break
            if not found:
                fp += 1
        fn += len(gt_boxes) - len(matched)

    precision = tp / (tp+fp+1e-9)
    recall = tp / (tp+fn+1e-9)
    f1 = 2*precision*recall / (precision+recall+1e-9)
    return precision, recall, f1


DATA_ROOT = Path("/content/drive/MyDrive/Store/MOT20/MOT20-clean")
VAL_IM = DATA_ROOT / "images" / "val"
VAL_LB = DATA_ROOT / "labels" / "val"
DETR_DIR = Path("/content/deform_detr_dets")  # your saved DETR JSONs

p, r, f1 = evaluate_detr(DETR_DIR, VAL_IM, VAL_LB, iou_thr=0.5)
print("Deformable DETR Validation Metrics:")
print(f"Precision = {p:.3f}")
print(f"Recall    = {r:.3f}")
print(f"F1 Score  = {f1:.3f}")


Deformable DETR Validation Metrics:
Precision = 0.952
Recall    = 0.074
F1 Score  = 0.137


WBF Fusion using inference of both yolov8 and deformable detr

In [ ]:
!pip install ensemble_boxes

In [ ]:
from ensemble_boxes import weighted_boxes_fusion
import torch, json
from pathlib import Path
from PIL import Image

# --- Paths ---
OUT_DIR = Path("/content/deform_detr_dets")        # Your current JSON detections
FUSED_OUT_DIR = Path("/content/deform_detr_dets_wbf")
FUSED_OUT_DIR.mkdir(parents=True, exist_ok=True)

VAL_IMG_DIR = Path("/content/drive/MyDrive/Store/MOT20/MOT20-clean/images/val")

# WBF parameters
IOU_THRESHOLD = 0.5
SCORE_THRESHOLD = 0.05  # same as inference threshold

# Helper to convert boxes to [0,1] normalized format
def normalize_boxes(boxes, width, height):
    norm_boxes = []
    for x1, y1, x2, y2, score, cls in boxes:
        norm_boxes.append([
            x1 / width,
            y1 / height,
            x2 / width,
            y2 / height
        ])
    return norm_boxes

# Process each JSON file
for json_file in OUT_DIR.glob("*.json"):
    with open(json_file, "r") as f:
        dets = json.load(f)  # [[x1,y1,x2,y2,score,cls], ...]

    if len(dets) == 0:
        fused_dets = []
    else:
        # Extract sequence name and frame number
        seq_name, frame_num = json_file.stem.split("_")  # e.g., MOT20-05, 000001
        img_path = VAL_IMG_DIR / seq_name / f"{frame_num}.jpg"

        # Open image to get width and height
        if not img_path.exists():
            print(f"Warning: Image not found {img_path}, skipping.")
            continue
        img = Image.open(img_path)
        W, H = img.width, img.height
        img.close()

        # Normalize boxes for WBF
        boxes = normalize_boxes(dets, W, H)
        scores = [d[4] for d in dets]
        labels = [int(d[5]) for d in dets]

        # WBF expects lists of predictions (for ensembles), here only 1 model
        boxes_list = [boxes]
        scores_list = [scores]
        labels_list = [labels]

        # Apply Weighted Boxes Fusion
        fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            iou_thr=IOU_THRESHOLD, skip_box_thr=SCORE_THRESHOLD
        )

        # Convert fused boxes back to absolute coordinates
        fused_dets = []
        for box, score, label in zip(fused_boxes, fused_scores, fused_labels):
            x1 = box[0] * W
            y1 = box[1] * H
            x2 = box[2] * W
            y2 = box[3] * H
            fused_dets.append([x1, y1, x2, y2, score, int(label)])

    # Save fused results
    out_file = FUSED_OUT_DIR / json_file.name
    with open(out_file, "w") as f:
        json.dump(fused_dets, f)

print("WBF fusion complete. Results saved to:", FUSED_OUT_DIR)

import shutil
from pathlib import Path

FUSED_OUT_DIR = Path("/content/deform_detr_dets_wbf")
DRIVE_OUT = Path("/content/drive/MyDrive/Store/deform_detr_dets_wbf")

# Remove old folder if exists, then copy new one
if DRIVE_OUT.exists():
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(FUSED_OUT_DIR, DRIVE_OUT)

print("Copied WBF fused results to Drive at:", DRIVE_OUT)


WBF fusion complete. Results saved to: /content/deform_detr_dets_wbf
Copied WBF fused results to Drive at: /content/drive/MyDrive/Store/deform_detr_dets_wbf


In [ ]:
from ensemble_boxes import weighted_boxes_fusion
import json
from pathlib import Path
from PIL import Image

# --- Paths ---
DETR_DIR = Path("/content/drive/MyDrive/deform_detr_dets2")
YOLO_DIR = Path("/content/drive/MyDrive/yolo_inf")
IMG_ROOT = Path("/content/drive/MyDrive/MOT20-clean/images")
FUSED_OUT_DIR = Path("/content/drive/MyDrive/deform_detr_dets_wbf2")
FUSED_OUT_DIR.mkdir(parents=True, exist_ok=True)

# WBF parameters
IOU_THRESHOLD = 0.5
SCORE_THRESHOLD = 0.05

# --- Helpers ---

def normalize_boxes(boxes, width, height):
    """
    Normalize bounding boxes to [0,1].
    Accepts boxes with at least 4 elements (x1, y1, x2, y2)
    Optionally followed by score and label.
    """
    return [[b[0] / width, b[1] / height, b[2] / width, b[3] / height] for b in boxes]

def get_scores_labels(boxes):
    """
    Safely extract scores and labels from boxes.
    Defaults: score=1.0, label=0 if missing.
    """
    scores = [b[4] if len(b) > 4 else 1.0 for b in boxes]
    labels = [int(b[5]) if len(b) > 5 else 0 for b in boxes]
    return scores, labels

def find_image_map(img_root):
    """
    Build a map: (SEQ_NAME_UPPER, FRAME_ID) -> image path
    Scans both train/val splits recursively.
    """
    frame_map = {}
    for split in ["train", "val"]:
        split_dir = img_root / split
        if not split_dir.exists():
            continue
        for seq_dir in split_dir.iterdir():
            if not seq_dir.is_dir():
                continue
            seq_name = seq_dir.name.upper()
            for img_file in seq_dir.glob("*.jpg"):
                frame_id = img_file.stem
                frame_map[(seq_name, frame_id)] = img_file
    return frame_map

# --- Build frame map once ---
frame_map = find_image_map(IMG_ROOT)

# --- Main Fusion Loop ---
for seq_dir in DETR_DIR.iterdir():
    if not seq_dir.is_dir():
        continue
    seq_name = seq_dir.name.upper()
    yolo_seq_dir = YOLO_DIR / seq_dir.name
    fused_seq_dir = FUSED_OUT_DIR / seq_dir.name
    fused_seq_dir.mkdir(parents=True, exist_ok=True)

    if not yolo_seq_dir.exists():
        print(f"⚠️ No YOLO results for {seq_dir.name}, skipping sequence.")
        continue

    for json_file in sorted(seq_dir.glob("*.json")):
        frame_id = json_file.stem
        yolo_file = yolo_seq_dir / f"{frame_id}.json"

        if not yolo_file.exists():
            print(f"⚠️ Missing YOLO file {yolo_file}, skipping frame.")
            continue

        img_path = frame_map.get((seq_name, frame_id))
        if img_path is None:
            print(f"⚠️ Missing image for frame {frame_id} in sequence {seq_name}, skipping.")
            continue

        W, H = Image.open(img_path).size

        # Load detections
        with open(json_file, "r") as f:
            dets_detr = json.load(f)
        with open(yolo_file, "r") as f:
            dets_yolo = json.load(f)

        # If both empty → save empty
        if len(dets_detr) == 0 and len(dets_yolo) == 0:
            fused_dets = []
        else:
            # DETR
            boxes_detr = normalize_boxes(dets_detr, W, H)
            scores_detr, labels_detr = get_scores_labels(dets_detr)

            # YOLO
            boxes_yolo = normalize_boxes(dets_yolo, W, H)
            scores_yolo, labels_yolo = get_scores_labels(dets_yolo)

            # Weighted Boxes Fusion
            fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
                [boxes_detr, boxes_yolo],
                [scores_detr, scores_yolo],
                [labels_detr, labels_yolo],
                iou_thr=IOU_THRESHOLD,
                skip_box_thr=SCORE_THRESHOLD
            )

            fused_dets = [
                [x1 * W, y1 * H, x2 * W, y2 * H, float(score), int(label)]
                for box, score, label in zip(fused_boxes, fused_scores, fused_labels)
                for x1, y1, x2, y2 in [box]
            ]

        # Save fused JSON
        out_file = fused_seq_dir / f"{frame_id}.json"
        with open(out_file, "w") as f:
            json.dump(fused_dets, f)

print("✅ WBF fusion complete. Results saved to:", FUSED_OUT_DIR)



# BYTE TRACK


In [ ]:
!pip install supervision==0.25.0
!pip install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.5/181.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.1 MB/s eta 0:00:00


In [ ]:
import os
import json
import numpy as np
from pathlib import Path
import supervision as sv

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- Paths ---
WBF_DETS_DIR = Path("/content/drive/MyDrive/Store/deform_detr_dets_wbf")   # input JSON files (already in Colab workspace)
NPY_OUT_DIR = Path("/content/drive/MyDrive/Store/byte_track_dets_npy")  # output .npy files on Google Drive
NPY_OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Initialize tracker ---
tracker = sv.ByteTrack()

# --- Loop over all JSON files ---
for json_file in sorted(WBF_DETS_DIR.glob("*.json")):
    with open(json_file, "r") as f:
        dets = json.load(f)  # [[x1,y1,x2,y2,score,cls], ...]

    dets_array = np.empty((0, 6))  # default empty

    if len(dets) > 0:
        xyxy = np.array([d[:4] for d in dets], dtype=np.float32)
        conf = np.array([d[4] for d in dets], dtype=np.float32)
        cls_id = np.array([d[5] for d in dets], dtype=np.int32)

        detections = sv.Detections(xyxy=xyxy, confidence=conf, class_id=cls_id)
        tracks = tracker.update_with_detections(detections)

        if len(tracks) > 0:
            dets_array = np.hstack([
                tracks.xyxy,                      # [x1,y1,x2,y2]
                tracks.confidence[:, None],       # confidence
                tracks.tracker_id[:, None]        # track ID
            ])

    # save with same frame name, but on Drive
    out_file = NPY_OUT_DIR / (json_file.stem + ".npy")
    np.save(out_file, dets_array)

print("✅ All frames processed. NPY files saved in:", NPY_OUT_DIR)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ All frames processed. NPY files saved in: /content/drive/MyDrive/Store/byte_track_dets_npy


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# --- Paths ---
NPY_IN_DIR = Path("/content/drive/MyDrive/Store/byte_track_dets_npy")   # input: raw ByteTrack .npy files
CSV_OUT_FILE = Path("/content/drive/MyDrive/Store/people_count.csv")    # output: CSV file

# --- Collect counts ---
records = []

for npy_file in sorted(NPY_IN_DIR.glob("*.npy")):
    frame_id = npy_file.stem  # use filename as frame ID
    dets_array = np.load(npy_file)

    num_people = len(dets_array)  # number of rows = number of people
    records.append({"frame_id": frame_id, "num_people": num_people})

# --- Save to CSV ---
df = pd.DataFrame(records)
df.to_csv(CSV_OUT_FILE, index=False)

print(f"✅ People count saved to {CSV_OUT_FILE}")


✅ People count saved to /content/drive/MyDrive/Store/people_count.csv


In [ ]:
import numpy as np
from pathlib import Path

# --- Paths ---
NPY_IN_DIR = Path("/content/drive/MyDrive/Store/byte_track_dets_npy")   # input: raw ByteTrack .npy files
FEATURE_OUT_FILE = Path("/content/drive/MyDrive/Store/byte_track_features.npy")  # output: training-ready features

features = []   # list of [num_people, mean_area, mean_conf]

for npy_file in sorted(NPY_IN_DIR.glob("*.npy")):
    dets_array = np.load(npy_file)  # [x1,y1,x2,y2,conf,track_id]

    if len(dets_array) > 0:
        num_people = len(dets_array)
        areas = (dets_array[:, 2] - dets_array[:, 0]) * (dets_array[:, 3] - dets_array[:, 1])
        mean_area = areas.mean()
        mean_conf = dets_array[:, 4].mean()
    else:
        num_people, mean_area, mean_conf = 0, 0, 0

    features.append([num_people, mean_area, mean_conf])

features = np.array(features, dtype=np.float32)
np.save(FEATURE_OUT_FILE, features)

print(f" Training features saved to {FEATURE_OUT_FILE}")


 Training features saved to /content/drive/MyDrive/Store/byte_track_features.npy


In [ ]:
import os
import json
import numpy as np
from pathlib import Path
import supervision as sv

# --- Paths ---
WBF_DETS_DIR = Path("/content/drive/MyDrive/deform_detr_dets_wbf2")
NPY_OUT_DIR = Path("/content/drive/MyDrive/byte_track_dets_npy2")
NPY_OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Loop over each MOT20 sequence ---
for seq_dir in sorted(WBF_DETS_DIR.iterdir()):
    if not seq_dir.is_dir():
        continue  # skip files

    seq_name = seq_dir.name
    print(f"Processing sequence: {seq_name}")

    # Make output subdir
    seq_out_dir = NPY_OUT_DIR / seq_name
    seq_out_dir.mkdir(parents=True, exist_ok=True)

    # Reset tracker for each sequence
    tracker = sv.ByteTrack()

    # Loop over all JSONs inside this sequence
    for json_file in sorted(seq_dir.glob("*.json")):
        with open(json_file, "r") as f:
            dets = json.load(f)

        dets_array = np.empty((0, 6))

        if len(dets) > 0:
            xyxy = np.array([d[:4] for d in dets], dtype=np.float32)
            conf = np.array([d[4] for d in dets], dtype=np.float32)
            cls_id = np.array([d[5] for d in dets], dtype=np.int32)

            detections = sv.Detections(xyxy=xyxy, confidence=conf, class_id=cls_id)
            tracks = tracker.update_with_detections(detections)

            if len(tracks) > 0:
                dets_array = np.hstack([
                    tracks.xyxy,
                    tracks.confidence[:, None],
                    tracks.tracker_id[:, None]
                ])

        # Save .npy with same frame name
        out_file = seq_out_dir / (json_file.stem + ".npy")
        np.save(out_file, dets_array)

    print(f"Finished {seq_name}, saved to {seq_out_dir}")

print("All sequences done. NPY files are in:", NPY_OUT_DIR)
